In [ ]:
#just loading the train,test,val.csv after uploading original one
import pandas as pd

train = pd.read_csv("Train.csv")
val   = pd.read_csv("Val.csv")
test  = pd.read_csv("Test.csv")

print("Train shape:", train.shape)
print("Columns:", train.columns)
train.head(3)

In [ ]:
#converting to 0,1 only to train,test,val and storing them in Train_binary.csv,Test_binary.csv....
import pandas as pd

train = pd.read_csv("Train.csv")
val   = pd.read_csv("Val.csv")
test  = pd.read_csv("Test.csv")

TEXT_COL  = "Data"
LABEL_COL = "Label"

def convert_to_binary(df):
    df = df.copy()

    # Keep only Positive (1) and Negative (2)
    df = df[df[LABEL_COL].isin([1, 2])]

    # Map to binary
    df[LABEL_COL] = df[LABEL_COL].replace({
        2: 0,   # Negative
        1: 1    # Positive
    })

    return df[[TEXT_COL, LABEL_COL]]

train_bin = convert_to_binary(train)
val_bin   = convert_to_binary(val)
test_bin  = convert_to_binary(test)

print("Train:", train_bin.shape, train_bin[LABEL_COL].value_counts())
print("Val  :", val_bin.shape,   val_bin[LABEL_COL].value_counts())
print("Test :", test_bin.shape,  test_bin[LABEL_COL].value_counts())

# Save new files
train_bin.to_csv("Train_binary.csv", index=False)
val_bin.to_csv("Val_binary.csv", index=False)
test_bin.to_csv("Test_binary.csv", index=False)

In [ ]:
for lab in sorted(train_bin[LABEL_COL].unique()):
    print("\n" + "="*60)
    print(f"BINARY LABEL = {lab}  ({'NEG' if lab==0 else 'POS'})")
    samples = train_bin[train_bin[LABEL_COL] == lab][[TEXT_COL, LABEL_COL]].head(5)
    for i, row in samples.iterrows():
        print(f"- ({row[LABEL_COL]}) {row[TEXT_COL]}")

In [ ]:
#checking split 80/10/10
import pandas as pd

train = pd.read_csv("Train.csv")
val   = pd.read_csv("Val.csv")
test  = pd.read_csv("Test.csv")

total = len(train) + len(val) + len(test)

print("Sizes:")
print("Train:", len(train), f"({len(train)/total:.2%})")
print("Val  :", len(val),   f"({len(val)/total:.2%})")
print("Test :", len(test),  f"({len(test)/total:.2%})")

In [ ]:
#checking label distribution in each split
LABEL_COL = "Label"

def dist(df, name):
    print(f"\n{name} label distribution:")
    print(df[LABEL_COL].value_counts())
    print(df[LABEL_COL].value_counts(normalize=True).round(3))

dist(train, "Train")
dist(val,   "Val")
dist(test,  "Test")

In [ ]:
#Zero-shot evaluation code (Accuracy, F1, Confusion Matrix + save result) --- no fine tuning
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import os

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# 1) Load binary test
test_df = pd.read_csv("Test_binary.csv")   # Data, Label (0=NEG, 1=POS)
test_df = test_df.rename(columns={"Data": "text", "Label": "label"})
test_df["text"] = test_df["text"].astype(str)
test_df["label"] = test_df["label"].astype(int)

# 2) HF dataset
test_ds = Dataset.from_pandas(test_df[["text", "label"]])

# 3) Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True)

test_ds = test_ds.map(tok, batched=True)
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# 4) Load model (3-class head: neg/neu/pos)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# 5) Predict (NO training)
trainer = Trainer(model=model, data_collator=collator)
pred = trainer.predict(test_ds)

logits = pred.predictions
pred_3class = np.argmax(logits, axis=1)  # 0=neg, 1=neu, 2=pos

# 6) Convert to binary (neutral -> negative)
pred_binary = np.where(pred_3class == 2, 1, 0)

y_true = test_df["label"].values

# 7) Metrics
acc = accuracy_score(y_true, pred_binary)
f1  = f1_score(y_true, pred_binary)
cm  = confusion_matrix(y_true, pred_binary)

print("Zero-shot Accuracy:", acc)
print("Zero-shot F1:", f1)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_true, pred_binary, target_names=["NEG", "POS"]))

# 8) Save results
os.makedirs("results", exist_ok=True)
with open("results/sentnob_zero_shot.txt", "w", encoding="utf-8") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write("DATASET: SentNoB (binary: neg/pos; neutral removed)\n\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"F1-score: {f1:.4f}\n\n")
    f.write("Confusion Matrix [ [TN FP], [FN TP] ]:\n")
    f.write(str(cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_true, pred_binary, target_names=["NEG", "POS"]))

In [ ]:
#fine-tune for train/test/val.csv
import os
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# -----------------------
# 0) Fixed settings
# -----------------------
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
SEED = 42

set_seed(SEED)
os.makedirs("results", exist_ok=True)

# -----------------------
# 1) Load binary CSVs
# -----------------------
def load_binary_csv(path):
    df = pd.read_csv(path)  # columns: Data, Label
    df = df.rename(columns={"Data": "text", "Label": "label"})
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)
    return df[["text", "label"]]

train_df = load_binary_csv("Train_binary.csv")
val_df   = load_binary_csv("Val_binary.csv")
test_df  = load_binary_csv("Test_binary.csv")

print("Train:", train_df.shape, train_df["label"].value_counts().to_dict())
print("Val  :", val_df.shape,   val_df["label"].value_counts().to_dict())
print("Test :", test_df.shape,  test_df["label"].value_counts().to_dict())

# -----------------------
# 2) Convert to HF Dataset
# -----------------------
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

# -----------------------
# 3) Tokenize
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# -----------------------
# 4) Load model for binary fine-tuning
# -----------------------
# NOTE: we set num_labels=2 because OUR task is binary now.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# -----------------------
# 5) Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)  # binary f1 (POS=1)
    }

# -----------------------
# 6) TrainingArguments (your fixed protocol)
# -----------------------
args = TrainingArguments(
    output_dir="xlmr_sentnob_finetuned",
    eval_strategy="epoch",          # ✅ changed
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=SEED,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

# -----------------------
# 7) Train
# -----------------------
trainer.train()

# -----------------------
# 8) Evaluate on test + Confusion Matrix
# -----------------------
pred = trainer.predict(test_ds)
logits = pred.predictions
y_pred = np.argmax(logits, axis=1)
y_true = test_df["label"].values

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred)

print("\nFINAL TEST RESULTS")
print("Accuracy:", acc)
print("F1:", f1)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["NEG", "POS"]))

# -----------------------
# 9) Save results
# -----------------------
with open("results/sentnob_finetuned.txt", "w", encoding="utf-8") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write("SETTING: Fine-tuned (binary: 0=NEG, 1=POS)\n")
    f.write(f"Max_len={MAX_LEN}, Batch={BATCH_SIZE}, Epochs={EPOCHS}, LR={LR}, Seed={SEED}\n\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"F1-score: {f1:.4f}\n\n")
    f.write("Confusion Matrix [ [TN FP], [FN TP] ]:\n")
    f.write(str(cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_true, y_pred, target_names=["NEG", "POS"]))

print("\nSaved to: results/sentnob_finetuned.txt")

In [ ]:
#checking mix of english word
import re

def code_mix_ratio_any_english(texts):
    total_words = 0
    english_like = 0

    for text in texts:
        words = str(text).split()
        total_words += len(words)
        for w in words:
            # counts token if it contains at least one English letter
            if re.search(r'[A-Za-z]', w):
                english_like += 1

    return english_like / total_words if total_words > 0 else 0

ratio2 = code_mix_ratio_any_english(train_df["text"])
print("SentNoB Code-Mix Ratio (any English char):", ratio2)

In [ ]:
#Misclassified samples for ZERO-SHOT model
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from transformers import DataCollatorWithPadding

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# 1) Load test data
df = pd.read_csv("Test_binary.csv")  # Data, Label
df = df.rename(columns={"Data": "text", "Label": "y_true"})
df["text"] = df["text"].astype(str)
df["y_true"] = df["y_true"].astype(int)

# 2) HF dataset
test_ds = Dataset.from_pandas(df[["text", "y_true"]])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

test_ds = test_ds.map(tok, batched=True)
test_ds.set_format("torch", columns=["input_ids", "attention_mask"])

# 3) Load model (3-class) + predict
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
trainer = Trainer(model=model, data_collator=collator)

pred = trainer.predict(test_ds)
logits = torch.tensor(pred.predictions)

# 4) Convert 3-class -> binary
# 0=neg,1=neu,2=pos => binary: pos->1, (neg/neu)->0
pred_3 = torch.argmax(logits, dim=1).numpy()
y_pred = np.where(pred_3 == 2, 1, 0)

# 5) Confidence for predicted class (optional)
probs = torch.softmax(logits, dim=1).numpy()
conf = probs.max(axis=1)

df["y_pred"] = y_pred
df["conf"] = conf
df["correct"] = (df["y_true"] == df["y_pred"])

# 6) Misclassified rows
mis = df[df["correct"] == False].copy()
print("Total test samples:", len(df))
print("Total misclassified:", len(mis))

# 7) Separate error types
false_pos = df[(df["y_true"] == 0) & (df["y_pred"] == 1)]  # NEG predicted POS
false_neg = df[(df["y_true"] == 1) & (df["y_pred"] == 0)]  # POS predicted NEG

print("\nFalse Positives (NEG→POS):", len(false_pos))
print("False Negatives (POS→NEG):", len(false_neg))

# 8) Print some examples (teacher evidence)
def show_examples(title, data, n=10):
    print("\n" + "="*70)
    print(title)
    print("="*70)
    sample = data.sample(n=min(n, len(data)), random_state=42)
    for i, row in sample.iterrows():
        true_label = "NEG" if row["y_true"] == 0 else "POS"
        pred_label = "NEG" if row["y_pred"] == 0 else "POS"
        print(f"\nTrue: {true_label} | Pred: {pred_label} | Confidence: {row['conf']:.3f}")
        print("Text:", row["text"])

show_examples("FALSE NEGATIVES (POS predicted as NEG) — main error in zero-shot", false_neg, n=10)
show_examples("FALSE POSITIVES (NEG predicted as POS)", false_pos, n=10)

# 9) Save misclassified examples as CSV
mis.to_csv("results/sentnob_zero_shot_misclassified.csv", index=False)
false_neg.to_csv("results/sentnob_zero_shot_false_negatives.csv", index=False)
false_pos.to_csv("results/sentnob_zero_shot_false_positives.csv", index=False)

print("\nSaved evidence CSVs in results/ folder.")

In [ ]:
!pip install sacremoses

In [ ]:
!ls -R .

In [ ]:
#misclassified ones for fined-tuned ones
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from transformers import DataCollatorWithPadding

CHECKPOINT_DIR = "xlmr_sentnob_finetuned/checkpoint-1818"
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# 1) Load test data
df = pd.read_csv("Test_binary.csv")
df = df.rename(columns={"Data":"text","Label":"y_true"})
df["text"] = df["text"].astype(str)
df["y_true"] = df["y_true"].astype(int)

test_ds = Dataset.from_pandas(df[["text","y_true"]])

# 2) Load tokenizer from original model (safe way)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 3) Load fine-tuned model
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

test_ds = test_ds.map(tok, batched=True)
test_ds.set_format("torch", columns=["input_ids","attention_mask"])

trainer = Trainer(model=model, data_collator=collator)
pred = trainer.predict(test_ds)

logits = torch.tensor(pred.predictions)
y_pred = torch.argmax(logits, dim=1).numpy()
conf = torch.softmax(logits, dim=1).max(dim=1).values.numpy()

df["y_pred"] = y_pred
df["conf"] = conf
df["correct"] = (df["y_true"] == df["y_pred"])

mis = df[df["correct"] == False]
false_pos = df[(df["y_true"] == 0) & (df["y_pred"] == 1)]
false_neg = df[(df["y_true"] == 1) & (df["y_pred"] == 0)]

print("Total misclassified (fine-tuned):", len(mis))
print("False Positives:", len(false_pos))
print("False Negatives:", len(false_neg))

def show_examples(title, data, n=10):
    print("\n" + "="*70)
    print(title)
    print("="*70)
    sample = data.sample(n=min(n, len(data)), random_state=42)
    for _, row in sample.iterrows():
        true_label = "NEG" if row["y_true"] == 0 else "POS"
        pred_label = "NEG" if row["y_pred"] == 0 else "POS"
        print(f"\nTrue: {true_label} | Pred: {pred_label} | Confidence: {row['conf']:.3f}")
        print("Text:", row["text"])

show_examples("FINE-TUNED FALSE NEGATIVES (POS→NEG)", false_neg, 10)
show_examples("FINE-TUNED FALSE POSITIVES (NEG→POS)", false_pos, 10)

mis.to_csv("results/sentnob_finetuned_misclassified.csv", index=False)
print("\nSaved: results/sentnob_finetuned_misclassified.csv")